# Kaggle SAM test_inference

This notebook runs a box-prompt SAM ViT-B inference and saves outputs to `outputs/`.

## 1. Load Dependencies and Verify Files

In [ ]:
import sys
from pathlib import Path
import subprocess

repo_root = Path.cwd()
print("repo_root:", repo_root)

# Install dependencies (quietly)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "opencv-python-headless",
        "matplotlib",
        "pillow",
    ],
    check=True,
)

try:
    import segment_anything  # noqa: F401
except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/facebookresearch/segment-anything.git",
        ],
        check=True,
    )

# Expected file locations
test_inference_path = repo_root / "test_inference.py"
test_box_path = repo_root / "test_box.txt"
image_path = repo_root / "notebooks" / "images" / "dog.jpg"
utils_path = repo_root / "sam_infer" / "utils.py"


def check_file(path: Path) -> None:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")


check_file(test_inference_path)
check_file(test_box_path)
check_file(image_path)
check_file(utils_path)

## 2. Download/Place SAM Checkpoint Weights

In [ ]:
import shutil
import urllib.request

checkpoint_name = "sam_vit_b_01ec64.pth"
checkpoint_dir = repo_root / "checkpoints"
checkpoint_path = checkpoint_dir / checkpoint_name
checkpoint_dir.mkdir(parents=True, exist_ok=True)

if not checkpoint_path.exists():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob(checkpoint_name))
        if matches:
            print("Copying checkpoint from:", matches[0])
            shutil.copyfile(matches[0], checkpoint_path)

    if not checkpoint_path.exists():
        url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
        print("Downloading:", url)
        urllib.request.urlretrieve(url, checkpoint_path)
else:
    print("Checkpoint already exists:", checkpoint_path)

print("Checkpoint path:", checkpoint_path)

## 3. Patch Python Path and Imports

In [ ]:
import textwrap
import urllib.request

sys.path.insert(0, str(repo_root))

sam_infer_dir = repo_root / "sam_infer"
sam_infer_dir.mkdir(parents=True, exist_ok=True)

if not utils_path.exists():
    utils_content = textwrap.dedent(
        """
        from __future__ import annotations

        from pathlib import Path
        from typing import Optional, Tuple

        import numpy as np
        import torch

        from segment_anything import SamPredictor, sam_model_registry


        class SAMInference:
            def __init__(
                self,
                checkpoint_path: Optional[str] = None,
                model_type: str = "vit_b",
                device: Optional[str] = None,
                output_dir: str = "outputs",
            ) -> None:
                if device is None:
                    device = "cuda" if torch.cuda.is_available() else "cpu"
                self.device = torch.device(device)

                if checkpoint_path is None:
                    checkpoint_path = str(
                        Path(__file__).resolve().parents[1]
                        / "checkpoints"
                        / "sam_vit_b_01ec64.pth"
                    )

                self.model = sam_model_registry[model_type](checkpoint=checkpoint_path)
                self.model.to(self.device)
                self.model.eval()
                self.predictor = SamPredictor(self.model)

                self.output_dir = Path(output_dir)

            def set_image(self, image: np.ndarray, image_format: str = "RGB") -> None:
                self.predictor.set_image(image, image_format=image_format)

            def predict(
                self,
                image: Optional[np.ndarray] = None,
                image_format: str = "RGB",
                point_coords: Optional[np.ndarray] = None,
                point_labels: Optional[np.ndarray] = None,
                box: Optional[np.ndarray] = None,
                mask_input: Optional[np.ndarray] = None,
                multimask_output: bool = True,
                return_logits: bool = False,
                save_prefix: str = "mask",
                save_png: bool = False,
            ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
                if image is not None:
                    self.set_image(image, image_format=image_format)
                if not self.predictor.is_image_set:
                    raise RuntimeError("Call set_image or pass image before predict.")

                if point_coords is not None:
                    point_coords = np.asarray(point_coords, dtype=np.float32)
                    if point_coords.ndim == 1:
                        point_coords = point_coords[None, :]
                if point_labels is not None:
                    point_labels = np.asarray(point_labels, dtype=np.int32)
                    if point_labels.ndim == 0:
                        point_labels = point_labels[None]
                if point_coords is not None and point_labels is None:
                    raise ValueError("point_labels must be provided with point_coords.")
                if box is not None:
                    box = np.asarray(box, dtype=np.float32)
                if mask_input is not None:
                    mask_input = np.asarray(mask_input, dtype=np.float32)
                    if mask_input.ndim == 2:
                        mask_input = mask_input[None, :, :]

                masks, scores, low_res = self.predictor.predict(
                    point_coords=point_coords,
                    point_labels=point_labels,
                    box=box,
                    mask_input=mask_input,
                    multimask_output=multimask_output,
                    return_logits=return_logits,
                )

                self.save_masks(masks, prefix=save_prefix, save_png=save_png)
                return masks, scores, low_res

            def save_masks(self, masks: np.ndarray, prefix: str = "mask", save_png: bool = False) -> None:
                self.output_dir.mkdir(parents=True, exist_ok=True)
                for i, mask in enumerate(masks):
                    if mask.dtype == np.bool_:
                        mask_to_save = mask.astype(np.uint8)
                    else:
                        mask_to_save = mask
                    np.save(self.output_dir / f"{prefix}_{i}.npy", mask_to_save)
                    if save_png:
                        if mask.dtype not in (np.bool_, np.uint8):
                            raise ValueError("save_png requires binary masks.")
                        try:
                            from PIL import Image
                        except ImportError as exc:
                            raise ImportError("save_png=True requires pillow.") from exc
                        Image.fromarray(mask_to_save.astype(np.uint8) * 255).save(
                            self.output_dir / f"{prefix}_{i}.png"
                        )
        """
    ).strip()
    utils_path.write_text(utils_content + "\n")

if not test_box_path.exists():
    test_box_path.write_text("76 45 146 87\n")

image_path.parent.mkdir(parents=True, exist_ok=True)
if not image_path.exists():
    url = "https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg"
    print("Downloading sample image:", url)
    urllib.request.urlretrieve(url, image_path)

if not test_inference_path.exists():
    test_inference_content = textwrap.dedent(
        """
        #!/usr/bin/env python3
        # Test script to run SAM inference with box prompt and visualize output.

        import sys
        from pathlib import Path

        import cv2
        import numpy as np

        sys.path.insert(0, str(Path(__file__).parent))

        from sam_infer.utils import SAMInference


        def visualize_mask(image: np.ndarray, masks: np.ndarray, output_path: Path) -> None:
            output_path.parent.mkdir(parents=True, exist_ok=True)

            overlay = image.copy().astype(float)
            colors = [
                [0, 255, 0],
                [255, 0, 0],
                [0, 0, 255],
            ]

            for i, mask in enumerate(masks):
                color = colors[i % len(colors)]
                overlay[mask > 0.5] = overlay[mask > 0.5] * 0.5 + np.array(color) * 0.5

            overlay = overlay.astype(np.uint8)
            cv2.imwrite(str(output_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
            print(f"Saved visualized output to {output_path}")


        def main() -> None:
            script_dir = Path(__file__).parent
            image_path = script_dir / "notebooks" / "images" / "dog.jpg"
            box_file = script_dir / "test_box.txt"
            output_dir = script_dir / "outputs"

            if not image_path.exists():
                raise FileNotFoundError(f"Image not found: {image_path}")
            if not box_file.exists():
                raise FileNotFoundError(f"Box file not found: {box_file}")

            image = cv2.imread(str(image_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            with open(box_file) as f:
                box_str = f.read().strip()
            box = np.array([float(x) for x in box_str.split()], dtype=np.float32)

            infer = SAMInference(output_dir=str(output_dir))
            masks, scores, _ = infer.predict(
                image=image,
                box=box,
                multimask_output=True,
                save_prefix="mask_box",
                save_png=True,
            )

            print("Masks shape:", masks.shape)
            print("IoU scores:", scores)
            visualize_mask(image, masks, output_dir / "visualization_box.png")


        if __name__ == "__main__":
            main()
        """
    ).strip()
    test_inference_path.write_text(test_inference_content + "\n")

print("Prepared local files for test_inference.")

## 4. Run test_inference.py

In [ ]:
import subprocess

subprocess.run([sys.executable, "test_inference.py"], check=True)

## 5. Display Output Artifacts

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

output_dir = repo_root / "outputs"
overlay_path = output_dir / "visualization_box.png"

print("Overlay:", overlay_path, overlay_path.exists())
if overlay_path.exists():
    overlay = cv2.imread(str(overlay_path))
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 6))
    plt.imshow(overlay)
    plt.axis("off")
    plt.show()

mask_files = sorted(output_dir.glob("mask_box_*.npy"))
print("Masks:", [p.name for p in mask_files])
for p in mask_files:
    mask = np.load(p)
    plt.figure(figsize=(4, 4))
    plt.imshow(mask, cmap="gray")
    plt.title(p.name)
    plt.axis("off")
    plt.show()